FINAL FITBIT

In [1]:

from notebookutils import mssparkutils
from pyspark.sql import functions as F
from collections import defaultdict


DIRS = [
  "abfss://e00a37d5-2770-4f5a-a766-c170cf67b6cf@onelake.dfs.fabric.microsoft.com/35e7fec8-9556-47ae-abef-86e2f2ef7593/Files/Fitabase Data 3.12.16-4.11.16",
  "abfss://e00a37d5-2770-4f5a-a766-c170cf67b6cf@onelake.dfs.fabric.microsoft.com/35e7fec8-9556-47ae-abef-86e2f2ef7593/Files/Fitabase Data 4.12.16-5.12.16",
]

spark.sql("SET spark.sql.shuffle.partitions=16")
spark.conf.set("spark.sql.adaptive.enabled", "true")

def ls_recursive(prefix: str):
    stack=[prefix]; out=[]
    while stack:
        p=stack.pop()
        try:
            for x in mssparkutils.fs.ls(p):
                if x.isDir:
                    stack.append(x.path)
                elif x.path.lower().endswith(".csv"):
                    out.append(x.path)
        except Exception as e:
            print(f" skip {p}: {e}")
    return out

def norm_table_name(fname: str) -> str:
    return fname.replace(".csv","").replace(" ","").lower()

# 1) find all CSVs across both folders
csvs=[]
for d in DIRS: 
    csvs.extend(ls_recursive(d))
print(f"Found {len(csvs)} CSV files")

# 2) group by filename (so same file from both folders gets merged)
groups=defaultdict(list)
for p in csvs:
    groups[p.split("/")[-1]].append(p)

print("Groups discovered:")
for fname, paths in sorted(groups.items()):
    print(f"{fname} -> {len(paths)} file(s)")

# 3) build one raw Delta table per filename (union columns by name, allowMissingColumns=True)
created=[]
for fname, paths in sorted(groups.items()):
    tname = norm_table_name(fname)
    print(f"\nMerging {fname}: {len(paths)} file(s) → table `{tname}`")
    dfs = [spark.read.option("header","true").csv(p) for p in paths]
    dfu = dfs[0]
    for d in dfs[1:]:
        dfu = dfu.unionByName(d, allowMissingColumns=True)
    dfu = dfu.dropDuplicates().repartition(16)  # row-level dedupe + consistent write parallelism

    spark.sql(f"DROP TABLE IF EXISTS {tname}")
    (dfu.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(tname))
    cnt = spark.table(tname).count()
    created.append((tname, cnt))
    print(f" wrote `{tname}` (rows: {cnt})")

print("\n=== RAW MERGED TABLES ===")
for t,c in created:
    print(f"{t}: {c}")


for probe in ["dailyactivity_merged","minutestepsnarrow_merged","heartrate_seconds_merged"]:
    try:
        print(f"\nSample: {probe}")
        spark.table(probe).show(5, truncate=False)
    except:
        pass


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 3, Finished, Available, Finished)

Found 29 CSV files
Groups discovered:
dailyActivity_merged.csv -> 2 file(s)
dailyCalories_merged.csv -> 1 file(s)
dailyIntensities_merged.csv -> 1 file(s)
dailySteps_merged.csv -> 1 file(s)
heartrate_seconds_merged.csv -> 2 file(s)
hourlyCalories_merged.csv -> 2 file(s)
hourlyIntensities_merged.csv -> 2 file(s)
hourlySteps_merged.csv -> 2 file(s)
minuteCaloriesNarrow_merged.csv -> 2 file(s)
minuteCaloriesWide_merged.csv -> 1 file(s)
minuteIntensitiesNarrow_merged.csv -> 2 file(s)
minuteIntensitiesWide_merged.csv -> 1 file(s)
minuteMETsNarrow_merged.csv -> 2 file(s)
minuteSleep_merged.csv -> 2 file(s)
minuteStepsNarrow_merged.csv -> 2 file(s)
minuteStepsWide_merged.csv -> 1 file(s)
sleepDay_merged.csv -> 1 file(s)
weightLogInfo_merged.csv -> 2 file(s)

Merging dailyActivity_merged.csv: 2 file(s) → table `dailyactivity_merged`
 wrote `dailyactivity_merged` (rows: 1397)

Merging dailyCalories_merged.csv: 1 file(s) → table `dailycalories_merged`
 wrote `dailycalories_merged` (rows: 940)

M

In [2]:

from pyspark.sql import functions as F

# 0) Parser + timezone for reproducible time handling

spark.sql("SET spark.sql.legacy.timeParserPolicy=LEGACY")
spark.conf.set("spark.sql.session.timeZone", "UTC")

# 1) Helpers
def to_ts_any(c):
    """
    Attempt several common timestamp formats found in the Fitbit CSVs.
    Returns a proper TimestampType column or null if none match.
    """
    return F.coalesce(
        F.to_timestamp(c, "M/d/yyyy H:mm"),
        F.to_timestamp(c, "M/d/yyyy h:mm a"),
        F.to_timestamp(c, "M/d/yyyy h:mm:ss a"),
        F.to_timestamp(c, "M/d/yyyy H:mm:ss"),
        F.to_timestamp(c, "MM/dd/yyyy H:mm"),
        F.to_timestamp(c, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(c)  # last-resort Spark parser
    )

def norm_null_str(col):
    """
    Cast to string and convert empty or 'NA'/'N/A'/'null' to real nulls.
    Keeps everything else as a trimmed string.
    """
    s = F.col(col).cast("string")
    return F.when(
        (F.trim(s) == "") | s.ilike("na") | s.ilike("n/a") | s.ilike("null"),
        None
    ).otherwise(F.trim(s))

# 2) Load and normalize minute-grain sources

# Steps per minute
ms = (spark.table("minutestepsnarrow_merged")
        .withColumn("Id", norm_null_str("Id").cast("string"))
        .withColumn("ActivityMinute", to_ts_any(norm_null_str("ActivityMinute")))
        .withColumn("Steps", norm_null_str("Steps").cast("double"))
        .dropna(subset=["Id","ActivityMinute"]))

# Calories per minute
mc = (spark.table("minutecaloriesnarrow_merged")
        .withColumn("Id", norm_null_str("Id").cast("string"))
        .withColumn("ActivityMinute", to_ts_any(norm_null_str("ActivityMinute")))
        .withColumn("Calories", norm_null_str("Calories").cast("double"))
        .dropna(subset=["Id","ActivityMinute"]))

# Intensity per minute
mi = (spark.table("minuteintensitiesnarrow_merged")
        .withColumn("Id", norm_null_str("Id").cast("string"))
        .withColumn("ActivityMinute", to_ts_any(norm_null_str("ActivityMinute")))
        .withColumn("Intensity", norm_null_str("Intensity").cast("double"))
        .dropna(subset=["Id","ActivityMinute"]))

# METs per minute
mm = (spark.table("minutemetsnarrow_merged")
        .withColumn("Id", norm_null_str("Id").cast("string"))
        .withColumn("ActivityMinute", to_ts_any(norm_null_str("ActivityMinute")))
        .withColumn("METs", norm_null_str("METs").cast("double"))
        .dropna(subset=["Id","ActivityMinute"]))

# Heart-rate seconds -> aggregate to per-minute stats
# We compute min, max, mean, stddev of HR within each minute.
hr_sec = (spark.table("heartrate_seconds_merged")
            .withColumn("Id", norm_null_str("Id").cast("string"))
            .withColumn("Time", to_ts_any(norm_null_str("Time")))
            .withColumn("Value", norm_null_str("Value").cast("double"))
            .dropna(subset=["Id","Time","Value"]))

hr_m = (hr_sec
        .withColumn("ActivityMinute", F.date_trunc("minute","Time"))
        .groupBy("Id","ActivityMinute")
        .agg(F.min("Value").alias("HR_Minute_Min"),
             F.max("Value").alias("HR_Minute_Max"),
             F.avg("Value").alias("HR_Minute_Mean"),
             # stddev can be null for single-sample minutes; coalesce to 0.0
             F.coalesce(F.stddev("Value"), F.lit(0.0)).alias("HR_Minute_StdDev")))

# 3) Build the minute-level master via OUTER joins
#    We retain a row if at least one of Steps or Calories is present,
#    then attach Intensity, METs, and HR stats when available.
minute_master = (ms.alias("s")
    .join(mc.alias("c"), ["Id","ActivityMinute"], "outer")
    .join(mi.alias("i"), ["Id","ActivityMinute"], "outer")
    .join(mm.alias("m"), ["Id","ActivityMinute"], "outer")
    .join(hr_m.alias("h"), ["Id","ActivityMinute"], "left")
    # Drop minutes that have neither Steps nor Calories (carry little signal)
    .dropna(subset=["Steps","Calories"], how="all")
    # Useful time derivations
    .withColumn("day",  F.to_date("ActivityMinute"))
    .withColumn("hour", F.date_trunc("hour","ActivityMinute"))
    .dropDuplicates(["Id","ActivityMinute"])
)

# 4) Persist as Delta table (schema overwrite allowed for repeatable runs)
spark.sql("DROP TABLE IF EXISTS final__activity_minute_master")
(minute_master.write
   .mode("overwrite")
   .option("overwriteSchema","true")
   .format("delta")
   .saveAsTable("final__activity_minute_master"))

# 5) Light QA prints (kept inexpensive)
n_master = spark.table("final__activity_minute_master").count()
print(f"final__activity_minute_master rows: {n_master}")

spark.table("final__activity_minute_master") \
     .select("Id","ActivityMinute","Steps","Calories","Intensity","METs","HR_Minute_Mean","HR_Minute_StdDev") \
     .orderBy("Id","ActivityMinute") \
     .show(5, truncate=False)


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 4, Finished, Available, Finished)

final__activity_minute_master rows: 1390080
+----------+-------------------+-----+-----------------+---------+----+--------------+----------------+
|Id        |ActivityMinute     |Steps|Calories         |Intensity|METs|HR_Minute_Mean|HR_Minute_StdDev|
+----------+-------------------+-----+-----------------+---------+----+--------------+----------------+
|1503960366|2016-03-12 01:00:00|16.0 |2.23243999481201 |0.0      |10.0|NULL          |NULL            |
|1503960366|2016-03-12 01:01:00|0.0  |0.797299981117249|0.0      |10.0|NULL          |NULL            |
|1503960366|2016-03-12 01:02:00|0.0  |0.956759989261627|0.0      |10.0|NULL          |NULL            |
|1503960366|2016-03-12 01:03:00|0.0  |0.797299981117249|0.0      |12.0|NULL          |NULL            |
|1503960366|2016-03-12 01:04:00|0.0  |0.956759989261627|0.0      |10.0|NULL          |NULL            |
+----------+-------------------+-----+-----------------+---------+----+--------------+----------------+
only showing top 5 r

In [3]:


from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1) Select and cast features from the master
feat_cols = ["Steps","Calories","Intensity","METs","HR_Mean","HR_StdDev"]
m = spark.table("final__activity_minute_master").select(
    F.col("Id").cast("string").alias("Id"),
    "ActivityMinute",
    F.col("Steps").cast("double").alias("Steps"),
    F.col("Calories").cast("double").alias("Calories"),
    F.col("Intensity").cast("double").alias("Intensity"),
    F.col("METs").cast("double").alias("METs"),
    F.col("HR_Minute_Mean").cast("double").alias("HR_Mean"),
    F.col("HR_Minute_StdDev").cast("double").alias("HR_StdDev")
)

# 2) Keep minutes that have signal (
mv = m.dropna(subset=["Steps","Calories"], how="all")

# 3) Per-user median imputation with global fallback
w_user = Window.partitionBy("Id")

# Compute per-user medians for each feature
for c in feat_cols:
    mv = mv.withColumn(f"{c}__user_med", F.percentile_approx(F.col(c), 0.5, 10000).over(w_user))

# Compute one-row global medians
global_meds = (
    mv.agg(*[F.percentile_approx(c, 0.5, 10000).alias(c) for c in feat_cols])
      .collect()[0]
      .asDict()
)

for c in feat_cols:
    mv = mv.withColumn(
        c,
        F.when(F.col(c).isNull(), F.coalesce(F.col(f"{c}__user_med"), F.lit(float(global_meds[c]))))
         .otherwise(F.col(c))
    ).drop(f"{c}__user_med")

# 4) Final type enforcement and residual-null drop (defensive)
for c in feat_cols:
    mv = mv.withColumn(c, F.col(c).cast("double"))
mv = mv.dropna(subset=feat_cols, how="any")

# 5) Persist the model view
spark.sql("DROP TABLE IF EXISTS final__minute_modelview")
(mv.write
   .mode("overwrite")
   .option("overwriteSchema","true")
   .format("delta")
   .saveAsTable("final__minute_modelview"))

# 6) Light QA
rows = spark.table("final__minute_modelview").count()
print("final__minute_modelview rows:", rows)
spark.table("final__minute_modelview").select("Id","ActivityMinute", *feat_cols).show(5, truncate=False)


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 5, Finished, Available, Finished)

final__minute_modelview rows: 1390080
+----------+-------------------+-----+-----------------+---------+----+-------+------------------+
|Id        |ActivityMinute     |Steps|Calories         |Intensity|METs|HR_Mean|HR_StdDev         |
+----------+-------------------+-----+-----------------+---------+----+-------+------------------+
|2026352035|2016-03-12 01:20:00|0.0  |2.0152599811554  |0.0      |26.0|90.4   |1.0690449676496991|
|2026352035|2016-03-12 01:35:00|31.0 |0.775099992752075|0.0      |10.0|90.4   |1.0690449676496991|
|2026352035|2016-03-12 01:40:00|34.0 |2.63533997535706 |1.0      |34.0|90.4   |1.0690449676496991|
|2026352035|2016-03-12 01:42:00|0.0  |0.775099992752075|1.0      |34.0|90.4   |1.0690449676496991|
|2026352035|2016-03-12 01:49:00|0.0  |0.775099992752075|0.0      |10.0|90.4   |1.0690449676496991|
+----------+-------------------+-----+-----------------+---------+----+-------+------------------+
only showing top 5 rows



In [4]:

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans, BisectingKMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.functions import vector_to_array

# 1) Load model-ready features
mv = spark.table("final__minute_modelview").cache()
features = ["Steps","Calories","Intensity","METs","HR_Mean","HR_StdDev"]

# 2) Assemble + standardize
va = VectorAssembler(inputCols=features, outputCol="features_raw")
sc = StandardScaler(withMean=True, withStd=True, inputCol="features_raw", outputCol="features")
scaled = sc.fit(va.transform(mv)).transform(va.transform(mv)).cache()

# 3) Train comparison set (3 algos × K=3..8)
def silhouette(pred_df):
    ev = ClusteringEvaluator(predictionCol="prediction", featuresCol="features", metricName="silhouette")
    return ev.evaluate(pred_df)

results = []   # list of tuples: (algo, K, silhouette, fitted_model)

for k in range(3, 9):
    # KMeans
    km = KMeans(k=k, seed=42, featuresCol="features")
    km_model = km.fit(scaled)
    km_pred  = km_model.transform(scaled)
    results.append(("kmeans", k, silhouette(km_pred), km_model))

    # Bisecting KMeans
    bk = BisectingKMeans(k=k, seed=42, featuresCol="features")
    bk_model = bk.fit(scaled)
    bk_pred  = bk_model.transform(scaled)
    results.append(("bisecting", k, silhouette(bk_pred), bk_model))

    # Gaussian Mixture
    gm = GaussianMixture(k=k, seed=42, featuresCol="features")
    gm_model = gm.fit(scaled)
    gm_pred  = gm_model.transform(scaled)  # includes probabilities
    results.append(("gmm", k, silhouette(gm_pred), gm_model))

# 4) Persist leaderboard and pick best
leader_df = spark.createDataFrame(
    [(a, int(k), float(s)) for (a,k,s,_) in results],
    schema="algo string, k int, silhouette double"
).orderBy(F.desc("silhouette"))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_leaderboard")
leader_df.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_leaderboard")

best_algo, best_k, best_silhouette, best_model = sorted(results, key=lambda x: x[2], reverse=True)[0]
print(f"Best model: {best_algo}  K={best_k}  silhouette={best_silhouette:.4f}")

# 5) Save best assignments
best_pred = (best_model
             .transform(scaled)
             .select("Id","ActivityMinute","prediction")
             .withColumnRenamed("prediction","cluster"))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_assignments")
best_pred.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_assignments")

# 6) Per-cluster feature means (for human-readable profiles)
mv_join = mv.join(best_pred, ["Id","ActivityMinute"], "inner")
summary = (mv_join.groupBy("cluster")
           .agg(F.count("*").alias("rows"),
                F.avg("Steps").alias("avg_steps"),
                F.avg("Calories").alias("avg_calories"),
                F.avg("Intensity").alias("avg_intensity"),
                F.avg("METs").alias("avg_mets"),
                F.avg("HR_Mean").alias("avg_hr"),
                F.avg("HR_StdDev").alias("avg_hr_std"))
           .orderBy("cluster"))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_summary")
summary.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_summary")

# 7) PCA to 2D for plotting
pca = PCA(k=2, inputCol="features", outputCol="pca")
pca_model = pca.fit(scaled)
pca_df = (pca_model.transform(scaled)
          .select("Id","ActivityMinute","pca"))
pca_df = (pca_df
          .withColumn("pca_arr", vector_to_array("pca"))
          .withColumn("pc1", F.col("pca_arr")[0])
          .withColumn("pc2", F.col("pca_arr")[1])
          .drop("pca","pca_arr"))

viz = pca_df.join(best_pred, ["Id","ActivityMinute"], "inner")

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_pca2")
viz.select("Id","ActivityMinute","cluster","pc1","pc2") \
   .write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_pca2")

# Small random sample for lightweight scatter plots in notebook UI
spark.sql("DROP TABLE IF EXISTS final__minute_cluster_pca2_sample")
viz.select("Id","ActivityMinute","cluster","pc1","pc2") \
   .sample(withReplacement=False, fraction=0.02, seed=42) \
   .write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_pca2_sample")

# 8) Quick prints to verify
print("\n=== Top 10 models by silhouette ===")
spark.table("final__minute_cluster_leaderboard").show(10, truncate=False)

print("\n=== Cluster summary (means) ===")
spark.table("final__minute_cluster_summary").show(truncate=False)

print("\nPCA sample rows:", spark.table("final__minute_cluster_pca2_sample").count())


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 6, Finished, Available, Finished)

Best model: kmeans  K=5  silhouette=0.6482



=== Top 10 models by silhouette ===
+---------+---+------------------+
|algo     |k  |silhouette        |
+---------+---+------------------+
|kmeans   |5  |0.6481855272180913|
|gmm      |4  |0.5587146509043265|
|kmeans   |8  |0.5475903274600508|
|kmeans   |3  |0.5338371602728628|
|kmeans   |7  |0.5317837839739148|
|bisecting|8  |0.5273192356267434|
|bisecting|7  |0.5234627586579907|
|gmm      |3  |0.5197331098078872|
|kmeans   |4  |0.5168893960769729|
|bisecting|6  |0.5123604358414687|
+---------+---+------------------+
only showing top 10 rows


=== Cluster summary (means) ===
+-------+-------+------------------+------------------+-------------------+------------------+-----------------+------------------+
|cluster|rows   |avg_steps         |avg_calories      |avg_intensity      |avg_mets          |avg_hr           |avg_hr_std        |
+-------+-------+------------------+------------------+-------------------+------------------+-----------------+------------------+
|0      |38822  |1

In [7]:

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.functions import vector_to_array

#  0) Load model-ready minute view ----
mv = spark.table("final__minute_modelview").cache()

#  1) Assemble + scale once (reuse for sample & full) ----
features = ["Steps","Calories","Intensity","METs","HR_Mean","HR_StdDev"]
va = VectorAssembler(inputCols=features, outputCol="features_raw")
scaler = StandardScaler(withMean=True, withStd=True, inputCol="features_raw", outputCol="features")

va_df = va.transform(mv).cache()
scaler_model = scaler.fit(va_df)
scaled = scaler_model.transform(va_df).cache()

#  2) Quick KMeans sweep on a 15% sample for silhouette leaderboard ----
quick_sample = scaled.sample(False, 0.15, seed=42).cache()

def silhouette(pred_df):
    ev = ClusteringEvaluator(predictionCol="prediction", featuresCol="features", metricName="silhouette")
    return ev.evaluate(pred_df)

Ks = list(range(3, 9))   
results = []             

for k in Ks:
    km = KMeans(k=k, seed=42, featuresCol="features", initSteps=5, maxIter=25)
    km_model = km.fit(quick_sample)
    km_pred  = km_model.transform(quick_sample)
    results.append(("kmeans", k, 42, float(silhouette(km_pred)), km_model))

leader_df = (spark.createDataFrame(
                [(a, int(k), int(seed), float(s)) for (a,k,seed,s,_) in results],
                "algo string, k int, seed int, silhouette double")
             .orderBy(F.desc("silhouette")))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_leaderboard")
leader_df.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_leaderboard")

best_k = leader_df.first()["k"]
best_sil = leader_df.first()["silhouette"]
print(f"Best model (sample): kmeans  K={best_k}  silhouette={best_sil:.4f}")

# 3) Refit best K on the FULL dataset and write assignments ----
best_model_full = KMeans(k=best_k, seed=42, featuresCol="features", initSteps=5, maxIter=40).fit(scaled)

best_pred = (best_model_full
             .transform(scaled)
             .select("Id","ActivityMinute","prediction")
             .withColumnRenamed("prediction","cluster"))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_assignments")
best_pred.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_assignments")

# 4) Cluster summaries in original feature space (means per cluster) ----
mv_join = (mv.join(best_pred, ["Id","ActivityMinute"], "inner"))
summary = (mv_join.groupBy("cluster")
           .agg(F.count("*").alias("rows"),
                F.avg("Steps").alias("avg_steps"),
                F.avg("Calories").alias("avg_calories"),
                F.avg("Intensity").alias("avg_intensity"),
                F.avg("METs").alias("avg_mets"),
                F.avg("HR_Mean").alias("avg_hr"),
                F.avg("HR_StdDev").alias("avg_hr_std"))
           .orderBy("cluster"))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_summary")
summary.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_summary")

# 5) PCA scatter (2D) for visualization + a small sample table for dashboards ----
pca = PCA(k=2, inputCol="features", outputCol="pca")
pca_model = pca.fit(scaled)

pca_df = (pca_model.transform(scaled)
          .select("Id","ActivityMinute","pca")
          .withColumn("pca_arr", vector_to_array("pca"))
          .withColumn("pc1", F.col("pca_arr")[0])
          .withColumn("pc2", F.col("pca_arr")[1])
          .drop("pca","pca_arr"))

viz = pca_df.join(best_pred, ["Id","ActivityMinute"], "inner")

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_pca2")
viz.select("Id","ActivityMinute","cluster","pc1","pc2") \
   .write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_pca2")

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_pca2_sample")
viz.select("Id","ActivityMinute","cluster","pc1","pc2") \
   .sample(False, 0.02, seed=42) \
   .write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_pca2_sample")

# 6) Quick prints to verify outputs ----
print("\n=== Top 10 models by silhouette (sample) ===")
spark.table("final__minute_cluster_leaderboard").show(10, truncate=False)

print("\n=== Cluster summary (means) ===")
spark.table("final__minute_cluster_summary").show(truncate=False)

print("\nPCA sample rows:", spark.table("final__minute_cluster_pca2_sample").count())


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 9, Finished, Available, Finished)

Best model (sample): kmeans  K=5  silhouette=0.6507



=== Top 10 models by silhouette (sample) ===
+------+---+----+------------------+
|algo  |k  |seed|silhouette        |
+------+---+----+------------------+
|kmeans|5  |42  |0.6506579756186032|
|kmeans|4  |42  |0.6496320726256835|
|kmeans|8  |42  |0.5839521023598022|
|kmeans|7  |42  |0.5775633226384383|
|kmeans|6  |42  |0.5661743157382415|
|kmeans|3  |42  |0.536467416377929 |
+------+---+----+------------------+


=== Cluster summary (means) ===
+-------+------+------------------+------------------+--------------------+------------------+-----------------+------------------+
|cluster|rows  |avg_steps         |avg_calories      |avg_intensity       |avg_mets          |avg_hr           |avg_hr_std        |
+-------+------+------------------+------------------+--------------------+------------------+-----------------+------------------+
|0      |37169 |15.518388980064032|2.935040553932556 |0.6352874707417472  |26.26479055126584 |85.32148459990785|18.08025569645729 |
|1      |970730|0.8362

In [8]:

from pyspark.sql import functions as F

# 1) Load the latest cluster summary (means by cluster)
summary = spark.table("final__minute_cluster_summary")

# 2) Simple, transparent naming logic based on average activity levels
#    Tune thresholds if your data shifts (they’re intentionally coarse and explainable).
def name_expr():
    return (
        F.when( (F.col("avg_steps") >= 60) & (F.col("avg_mets") >= 30), "Workout")
         .when( (F.col("avg_steps") >= 12) & (F.col("avg_mets") >= 20), "Brisk")
         .when( (F.col("avg_steps") >= 6)  & (F.col("avg_mets") >= 18), "Moderate")
         .otherwise("Light")
    )

names = (summary
         .select("cluster")
         .distinct()
         .join(summary.select("cluster",
                              "avg_steps","avg_calories","avg_intensity","avg_mets",
                              "avg_hr","avg_hr_std"),
               on="cluster", how="left")
         .withColumn("cluster_name", name_expr())
         .select("cluster", "cluster_name")
         .orderBy("cluster"))

# 3) Make names safe for Delta pivots and BI tools (no spaces/punctuation)
names = names.withColumn("cluster_name", F.regexp_replace("cluster_name", r"[^A-Za-z0-9_]", "_"))

# 4) Persist names for downstream joins (hour-of-day mix, confusion matrix, dashboards)
spark.sql("DROP TABLE IF EXISTS final__minute_cluster_names")
names.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_names")

print("Cluster names:")
display(spark.table("final__minute_cluster_names"))

# 5) Optional: write a named summary table for easier reporting
named_summary = (summary.join(names, on="cluster", how="left")
                        .select("cluster","cluster_name","rows","avg_steps","avg_calories",
                                "avg_intensity","avg_mets","avg_hr","avg_hr_std")
                        .orderBy("cluster"))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_summary_named")
named_summary.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_summary_named")

print("Named summary:")
display(spark.table("final__minute_cluster_summary_named"))


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 10, Finished, Available, Finished)

Cluster names:


SynapseWidget(Synapse.DataFrame, 3d66805c-7091-4148-9e76-ffe471006575)

Named summary:


SynapseWidget(Synapse.DataFrame, 0ab2865f-46fd-49bc-a14a-f1f0790baa2b)

In [9]:

from pyspark.sql import functions as F, Window

# Source: best assignments + friendly names
assign = spark.table("final__minute_cluster_assignments")   # Id, ActivityMinute, cluster
names  = spark.table("final__minute_cluster_names")         # cluster, cluster_name

# 0..23 hour extraction and join names
by_min = (assign
    .withColumn("hour_of_day", F.hour("ActivityMinute"))     # 0..23
    .join(names, on="cluster", how="left"))

# Count minutes per hour per named cluster
by_hour = (by_min
    .groupBy("hour_of_day","cluster_name")
    .agg(F.count("*").alias("count")))

# Hour totals
totals = by_hour.groupBy("hour_of_day").agg(F.sum("count").alias("tot"))

# Percent share per hour per named cluster
dist_named = (by_hour.join(totals, "hour_of_day")
    .withColumn("pct", (F.col("count") * F.lit(100.0)) / F.col("tot"))
    .select("hour_of_day","cluster_name","count","pct")
    .orderBy("hour_of_day","cluster_name"))

# Persist for dashboards
spark.sql("DROP TABLE IF EXISTS final__minute_cluster_dist_by_hour_named")
dist_named.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_dist_by_hour_named")


w = Window.partitionBy("cluster_name").orderBy("hour_of_day").rowsBetween(-1, 1)
dist_smooth = (dist_named
    .withColumn("pct_smooth", F.avg("pct").over(w))
    .select("hour_of_day","cluster_name","pct_smooth")
    .orderBy("hour_of_day","cluster_name"))

spark.sql("DROP TABLE IF EXISTS final__minute_cluster_dist_by_hour_named_smooth")
dist_smooth.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_dist_by_hour_named_smooth")


print("Distinct hours present:", [r[0] for r in dist_named.select("hour_of_day").distinct().orderBy("hour_of_day").collect()])

display(spark.table("final__minute_cluster_dist_by_hour_named_smooth"))


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 11, Finished, Available, Finished)

Distinct hours present: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


SynapseWidget(Synapse.DataFrame, b86833b1-7f4e-4f09-aa39-4a91e1e8899f)

In [10]:

from pyspark.sql import functions as F, Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 7A) Build per-hour features
feat_cols = ["Steps","Calories","Intensity","METs","HR_Mean","HR_StdDev"]

mv = spark.table("final__minute_modelview") \
         .withColumn("hour", F.date_trunc("hour", "ActivityMinute"))

hour_feats = (mv.groupBy("Id","hour")
                .agg(*[F.avg(c).alias(f"avg_{c}") for c in feat_cols]))

# 7B) Build per-hour labels (majority cluster each hour, then shift to next hour)
assign = spark.table("final__minute_cluster_assignments") \
             .withColumn("hour", F.date_trunc("hour", "ActivityMinute"))

names  = spark.table("final__minute_cluster_names")

min_lbl = assign.join(names, on="cluster", how="left")

hour_lbl = (min_lbl.groupBy("Id","hour","cluster_name")
                  .agg(F.count("*").alias("n"))
                  .withColumn("rn", F.row_number().over(
                      Window.partitionBy("Id","hour").orderBy(F.desc("n"), F.asc("cluster_name"))
                  ))
                  .where(F.col("rn")==1)
                  .select("Id","hour", F.col("cluster_name").alias("label_curr")))

w_next = Window.partitionBy("Id").orderBy("hour")
hour_lbl = hour_lbl.withColumn("label_next", F.lead("label_curr").over(w_next))


hour_lbl = hour_lbl.dropna(subset=["label_next"])

#7C) Join features + label_next
data = (hour_feats.join(hour_lbl, ["Id","hour"], "inner")
                 .select("Id","hour","label_next", *[f"avg_{c}" for c in feat_cols]))

#7D) Train/test split & model
assembler = VectorAssembler(
    inputCols=[f"avg_{c}" for c in feat_cols],
    outputCol="features_raw"
)
scaler    = StandardScaler(withMean=True, withStd=True, inputCol="features_raw", outputCol="features")
lab_index = StringIndexer(inputCol="label_next", outputCol="label").fit(data)

rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            numTrees=120, maxDepth=12, seed=42)

pipe = Pipeline(stages=[assembler, scaler, lab_index, rf])

train, test = data.randomSplit([0.85, 0.15], seed=42)

model = pipe.fit(train)
pred  = model.transform(test)

# 7E) Metrics 
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

acc = acc_eval.evaluate(pred)
f1  = f1_eval.evaluate(pred)
print(f"[Next-hour RF] accuracy={acc:.3f}  F1={f1:.3f}")


idx_to_name = model.stages[2].labels
to_name = F.udf(lambda i: idx_to_name[int(i)] if i is not None else None, "string")

cm = (pred.select(to_name("label").alias("label_name"),
                  to_name("prediction").alias("pred_name"))
          .groupBy("label_name","pred_name")
          .agg(F.count("*").alias("n"))
     )

# Save raw counts
spark.sql("DROP TABLE IF EXISTS final__cm_counts")
cm.write.mode("overwrite").format("delta").saveAsTable("final__cm_counts")

# Row % (per true label)
row_tot = cm.groupBy("label_name").agg(F.sum("n").alias("row_n"))
cm_pct  = (cm.join(row_tot, "label_name")
             .withColumn("row_pct", (F.col("n")*100.0)/F.col("row_n")))

spark.sql("DROP TABLE IF EXISTS final__cm_rowpct")
cm_pct.write.mode("overwrite").format("delta").saveAsTable("final__cm_rowpct")

#7F) Pivoted (safe) versions for heatmaps
cm_safe = (cm_pct
    .withColumn("label_name_s", F.regexp_replace("label_name", r"[^A-Za-z0-9_]", "_"))
    .withColumn("pred_name_s",  F.regexp_replace("pred_name",  r"[^A-Za-z0-9_]", "_"))
)

# counts pivot
pivot_counts = (cm_safe.groupBy("label_name_s")
                        .pivot("pred_name_s")
                        .agg(F.first("n"))
                        .na.fill(0)
                        .orderBy("label_name_s"))

spark.sql("DROP TABLE IF EXISTS final__cm_counts_pivot")
pivot_counts.write.mode("overwrite").format("delta").saveAsTable("final__cm_counts_pivot")

# row % pivot
pivot_rowpct = (cm_safe.groupBy("label_name_s")
                        .pivot("pred_name_s")
                        .agg(F.first("row_pct"))
                        .orderBy("label_name_s"))

spark.sql("DROP TABLE IF EXISTS final__cm_rowpct_pivot")
pivot_rowpct.write.mode("overwrite").format("delta").saveAsTable("final__cm_rowpct_pivot")

print("Saved: final__cm_counts, final__cm_rowpct, final__cm_counts_pivot, final__cm_rowpct_pivot")

display(spark.table("final__cm_rowpct_pivot"))


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 12, Finished, Available, Finished)

[Next-hour RF] accuracy=0.940  F1=0.917
Saved: final__cm_counts, final__cm_rowpct, final__cm_counts_pivot, final__cm_rowpct_pivot


SynapseWidget(Synapse.DataFrame, ee678437-ffcb-4c58-811c-59431bdc8946)

In [11]:

from pyspark.sql import functions as F

cm = spark.table("final__cm_counts")

row_tot = cm.groupBy("label_name").agg(F.sum("n").alias("row_n"))
col_tot = cm.groupBy("pred_name").agg(F.sum("n").alias("col_n"))


tp = cm.where(F.col("label_name")==F.col("pred_name")) \
       .select(F.col("label_name").alias("cls"), F.col("n").alias("tp"))

per_class = (tp.join(row_tot, tp.cls==row_tot.label_name, "left")
               .join(col_tot, tp.cls==col_tot.pred_name, "left")
               .select(
                   "cls",
                   "tp",
                   F.col("row_n").alias("support"),
                   F.col("col_n").alias("pred_total"))
               .withColumn("precision", F.when(F.col("pred_total")>0, F.col("tp")/F.col("pred_total")).otherwise(F.lit(0.0)))
               .withColumn("recall",    F.when(F.col("support")>0, F.col("tp")/F.col("support")).otherwise(F.lit(0.0)))
               .withColumn("f1",        F.when((F.col("precision")+F.col("recall"))>0,
                                               2*F.col("precision")*F.col("recall")/(F.col("precision")+F.col("recall")))
                                       .otherwise(F.lit(0.0)))
            ).orderBy("cls")

display(per_class)


macro = per_class.agg(F.avg("precision").alias("macro_precision"),
                      F.avg("recall").alias("macro_recall"),
                      F.avg("f1").alias("macro_f1"))

weighted = per_class.agg(
    (F.sum(F.col("precision")*F.col("support"))/F.sum("support")).alias("weighted_precision"),
    (F.sum(F.col("recall")*F.col("support"))/F.sum("support")).alias("weighted_recall"),
    (F.sum(F.col("f1")*F.col("support"))/F.sum("support")).alias("weighted_f1"),
)

print("Macro averages:")
display(macro)
print("Weighted averages:")
display(weighted)


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, bee89104-a513-4d3f-8e32-959367821ba5)

Macro averages:


SynapseWidget(Synapse.DataFrame, 5770013f-0862-4ef0-abca-d19d604c4285)

Weighted averages:


SynapseWidget(Synapse.DataFrame, 7defef74-e507-40aa-9ad8-4da23fc0e702)

In [13]:
from pyspark.sql import functions as F, Window
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1) Load features and minute-level cluster assignments
mv = spark.table("final__minute_modelview").select(
    "Id","ActivityMinute","Steps","Calories","Intensity","METs","HR_Mean","HR_StdDev"
)
assign = spark.table("final__minute_cluster_assignments").select("Id","ActivityMinute","cluster")
names = spark.table("final__minute_cluster_names") \
            .select(F.col("cluster").cast("int").alias("cid"), "cluster_name")

#2) Label = cluster 60 minutes later (next-hour)
w = Window.partitionBy("Id").orderBy("ActivityMinute")
labels = assign.withColumn("label_cluster", F.lead("cluster", 60).over(w)) \
               .select("Id","ActivityMinute","label_cluster") \
               .dropna()

base = (mv.join(labels, ["Id","ActivityMinute"], "inner")
          .withColumn("label_cluster", F.col("label_cluster").cast("int")))

# Add readable label name
base = (base.join(names, base.label_cluster == names.cid, "left")
             .drop("cid")
             .withColumnRenamed("cluster_name","label_name"))

# 3) Assemble + scale features
feat_cols = ["Steps","Calories","Intensity","METs","HR_Mean","HR_StdDev"]
va = VectorAssembler(inputCols=feat_cols, outputCol="features_raw")
sc = StandardScaler(withMean=True, withStd=True, inputCol="features_raw", outputCol="features")

full = sc.fit(va.transform(base)).transform(va.transform(base)) \
         .select("Id","ActivityMinute","features","label_cluster","label_name")

# 4) Stratified split (~15% test)
fractions = {int(r["label_cluster"]): 0.15
             for r in full.select("label_cluster").distinct().collect()}

test_keys = (full.select("label_cluster","Id","ActivityMinute")
                 .sampleBy("label_cluster", fractions=fractions, seed=42))

test_raw  = full.join(test_keys, ["label_cluster","Id","ActivityMinute"], "inner").cache()
train_raw = full.join(test_keys, ["label_cluster","Id","ActivityMinute"], "left_anti").cache()

print("train_raw rows:", train_raw.count(), "   test_raw rows:", test_raw.count())


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 15, Finished, Available, Finished)

train_raw rows: 1179522    test_raw rows: 208458


In [16]:

from pyspark.sql import functions as F
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1) Inverse-frequency class weights computed on TRAIN set
freq = (train_raw.groupBy("label_cluster","label_name").count()
                   .withColumnRenamed("count","n"))
N = train_raw.count()
K = freq.count()
weights = freq.withColumn("weight", (F.lit(N) / (F.lit(K) * F.col("n"))))

train_w = (train_raw.join(weights.select("label_cluster","weight"),
                          on="label_cluster", how="left"))

# 2) Train weighted RF (multiclass)
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label_cluster",
    weightCol="weight",
    numTrees=120,
    maxDepth=14,
    subsamplingRate=0.8,
    featureSubsetStrategy="sqrt",
    seed=42
)
rf_model = rf.fit(train_w)

# 3) Evaluate on TEST
pred = rf_model.transform(test_raw).cache()

acc = MulticlassClassificationEvaluator(
    labelCol="label_cluster", predictionCol="prediction", metricName="accuracy"
).evaluate(pred)
f1 = MulticlassClassificationEvaluator(
    labelCol="label_cluster", predictionCol="prediction", metricName="f1"
).evaluate(pred)

print(f"[Weighted RF] accuracy={acc:.3f}  F1={f1:.3f}")

# 4) Confusion matrix (counts and row %), with readable names

from pyspark.sql import functions as F

# Make a tiny DF with the prediction as an INT key
p = (pred
     .select("label_name", F.col("prediction").cast("int").alias("pred_id"))
     .alias("p"))

# Names table with the same INT key
names = (spark.table("final__minute_cluster_names")
         .select(F.col("cluster").cast("int").alias("pred_id"),
                 F.col("cluster_name").alias("pred_name"))
         .alias("n"))

# Join on the key, then aggregate
cm = (p.join(names, on="pred_id", how="left")
        .select("label_name","pred_name"))

cm_counts = (cm.groupBy("label_name","pred_name")
               .count().withColumnRenamed("count","n"))

row_tot   = cm_counts.groupBy("label_name").agg(F.sum("n").alias("row_n"))
cm_rowpct = (cm_counts.join(row_tot, "label_name")
                        .withColumn("row_pct", (F.col("n")*100.0)/F.col("row_n"))
                        .select("label_name","pred_name","n","row_n","row_pct"))

spark.sql("DROP TABLE IF EXISTS final__cm_counts")
spark.sql("DROP TABLE IF EXISTS final__cm_rowpct")
cm_counts.write.mode("overwrite").format("delta").saveAsTable("final__cm_counts")
cm_rowpct.write.mode("overwrite").format("delta").saveAsTable("final__cm_rowpct")

# Optional pivots for heatmaps
pivot_counts = (cm_counts.groupBy("label_name")
                          .pivot("pred_name").agg(F.first("n"))
                          .fillna(0).orderBy("label_name"))
pivot_rowpct = (cm_rowpct.groupBy("label_name")
                           .pivot("pred_name").agg(F.first("row_pct"))
                           .orderBy("label_name"))

spark.sql("DROP TABLE IF EXISTS final__cm_counts_pivot")
spark.sql("DROP TABLE IF EXISTS final__cm_rowpct_pivot")
pivot_counts.write.mode("overwrite").format("delta").saveAsTable("final__cm_counts_pivot")
pivot_rowpct.write.mode("overwrite").format("delta").saveAsTable("final__cm_rowpct_pivot")

display(spark.table("final__cm_rowpct_pivot"))





# Persist tables (safe for Delta pivots)
spark.sql("DROP TABLE IF EXISTS final__cm_counts")
spark.sql("DROP TABLE IF EXISTS final__cm_rowpct")
cm_counts.write.mode("overwrite").format("delta").saveAsTable("final__cm_counts")
cm_rowpct.write.mode("overwrite").format("delta").saveAsTable("final__cm_rowpct")

# Optional pivots for heatmaps
pivot_counts = (cm_counts.groupBy("label_name")
                          .pivot("pred_name")
                          .agg(F.first("n"))
                          .fillna(0)
                          .orderBy("label_name"))
pivot_rowpct = (cm_rowpct.groupBy("label_name")
                           .pivot("pred_name")
                           .agg(F.first("row_pct"))
                           .orderBy("label_name"))

spark.sql("DROP TABLE IF EXISTS final__cm_counts_pivot")
spark.sql("DROP TABLE IF EXISTS final__cm_rowpct_pivot")
pivot_counts.write.mode("overwrite").format("delta").saveAsTable("final__cm_counts_pivot")
pivot_rowpct.write.mode("overwrite").format("delta").saveAsTable("final__cm_rowpct_pivot")

display(spark.table("final__cm_rowpct_pivot"))   # Heatmap: Rows=label_name, Cols=pred_name, Values=% 


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 18, Finished, Available, Finished)

[Weighted RF] accuracy=0.517  F1=0.579


SynapseWidget(Synapse.DataFrame, b688791f-b38a-479f-8825-0b1bd9918f0a)

SynapseWidget(Synapse.DataFrame, a4541c5c-fc0f-4f0b-909a-45bcc6df3fb4)

In [17]:

from pyspark.sql import functions as F

# Join the PCA sample with cluster names
pca_samp = spark.table("final__minute_cluster_pca2_sample")         # pc1, pc2, cluster
names    = spark.table("final__minute_cluster_names")               # cluster, cluster_name

pca_named = (pca_samp.join(names, on="cluster", how="left")
                     .withColumn("cluster_name_s",
                                 F.regexp_replace("cluster_name", r"[^A-Za-z0-9_]", "_"))
                     .select("Id","ActivityMinute","pc1","pc2","cluster","cluster_name_s"))

# Persist a clean, small scatter dataset
spark.sql("DROP TABLE IF EXISTS final__minute_cluster_pca2_named_sample")
(pca_named.write
          .mode("overwrite")
          .format("delta")
          .saveAsTable("final__minute_cluster_pca2_named_sample"))

print("Saved: final__minute_cluster_pca2_named_sample")

display(spark.table("final__minute_cluster_pca2_named_sample").limit(5000))


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 19, Finished, Available, Finished)

Saved: final__minute_cluster_pca2_named_sample


SynapseWidget(Synapse.DataFrame, 3c3cb65d-4915-4c5e-8435-7d1fe238f7e9)

In [18]:

from pyspark.sql import functions as F


# A) Hour-of-day cluster mix

preferred_hour_tbl = "final__minute_cluster_dist_by_hour_named_smooth"

try:
    hour_df = spark.table(preferred_hour_tbl)
except Exception:
  
    base  = spark.table("final__minute_cluster_dist_by_hour")  
    names = spark.table("final__minute_cluster_names")         
    hour_df = (base
        .join(names, "cluster", "left")
        .withColumn("cluster_name_s", F.regexp_replace("cluster_name", r"[^A-Za-z0-9_]", "_"))
        .groupBy("hour_of_day","cluster_name_s")
        .agg(F.sum("count").alias("minutes"),
             F.first("tot").alias("tot"))
        .withColumn("pct", (F.col("minutes")*100.0)/F.col("tot"))
        .orderBy("hour_of_day","cluster_name_s")
    )

print("Hour-of-day mix → use Chart: Stacked bar (X=hour_of_day, Legend=cluster_name_s, Value=minutes) or Heatmap (Value=pct).")
display(hour_df)


# B) Confusion matrix (row % and raw counts)

def get_cm_pivots():
 
    try:
        cm_rowpct_pivot = spark.table("final__cm_rowpct_pivot")
        cm_counts_pivot = spark.table("final__cm_counts_pivot")
        return cm_rowpct_pivot, cm_counts_pivot
    except Exception:
   
        cm_rowpct = spark.table("final__cm_rowpct")   
        cm_counts = spark.table("final__cm_counts")   

        cm_rowpct_s = (cm_rowpct
            .withColumn("label_name_s", F.regexp_replace("label_name", r"[^A-Za-z0-9_]", "_"))
            .withColumn("pred_name_s",  F.regexp_replace("pred_name",  r"[^A-Za-z0-9_]", "_")))

        cm_counts_s = (cm_counts
            .withColumn("label_name_s", F.regexp_replace("label_name", r"[^A-Za-z0-9_]", "_"))
            .withColumn("pred_name_s",  F.regexp_replace("pred_name",  r"[^A-Za-z0-9_]", "_")))

        cm_rowpct_pivot = (cm_rowpct_s
            .groupBy("label_name_s")
            .pivot("pred_name_s")
            .agg(F.first("row_pct"))
            .orderBy("label_name_s"))

        cm_counts_pivot = (cm_counts_s
            .groupBy("label_name_s")
            .pivot("pred_name_s")
            .agg(F.first("n"))
            .fillna(0)
            .orderBy("label_name_s"))


        spark.sql("DROP TABLE IF EXISTS final__cm_rowpct_pivot")
        spark.sql("DROP TABLE IF EXISTS final__cm_counts_pivot")
        cm_rowpct_pivot.write.mode("overwrite").format("delta").saveAsTable("final__cm_rowpct_pivot")
        cm_counts_pivot.write.mode("overwrite").format("delta").saveAsTable("final__cm_counts_pivot")

        return cm_rowpct_pivot, cm_counts_pivot

cm_rowpct_pivot, cm_counts_pivot = get_cm_pivots()

print("Confusion Matrix (row %) → Chart: Heatmap (Rows=true label, Cols=predicted, Value=%).")
display(cm_rowpct_pivot)

print("Confusion Matrix (counts) → Chart: Heatmap or Table.")
display(cm_counts_pivot)


# C) PCA scatter (2D sample)
pca_named = spark.table("final__minute_cluster_pca2_named_sample")  # Id, ActivityMinute, pc1, pc2, cluster, cluster_name_s
print("PCA scatter → Chart: Scatter (X=pc1, Y=pc2, Color/Legend=cluster_name_s).")
display(pca_named)


StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 20, Finished, Available, Finished)

Hour-of-day mix → use Chart: Stacked bar (X=hour_of_day, Legend=cluster_name_s, Value=minutes) or Heatmap (Value=pct).


SynapseWidget(Synapse.DataFrame, 7bc854cf-d810-40df-a3fc-774398f77144)

Confusion Matrix (row %) → Chart: Heatmap (Rows=true label, Cols=predicted, Value=%).


SynapseWidget(Synapse.DataFrame, 8ab12f48-e09f-4a06-9855-1f1e933743d6)

Confusion Matrix (counts) → Chart: Heatmap or Table.


SynapseWidget(Synapse.DataFrame, eaac4ae8-65f2-4f7e-813a-348240c3413e)

PCA scatter → Chart: Scatter (X=pc1, Y=pc2, Color/Legend=cluster_name_s).


SynapseWidget(Synapse.DataFrame, a2fd2ace-c42b-41c1-8489-cfdc78f37f35)

In [20]:

from pyspark.sql import functions as F


feat = ["Steps","Calories","Intensity","METs","HR_Mean","HR_StdDev"]

mv     = spark.table("final__minute_modelview").select("Id","ActivityMinute", *feat)
assign = spark.table("final__minute_cluster_assignments").select("Id","ActivityMinute","cluster")
names  = spark.table("final__minute_cluster_names").select("cluster","cluster_name")


df = mv.join(assign, ["Id","ActivityMinute"], "inner")

# 1) Global moments (mean/std) for effect-size style deltas
glob = df.agg(
    *[F.avg(c).alias(f"{c}_gmean") for c in feat],
    *[F.stddev(c).alias(f"{c}_gsd") for c in feat]
)

# 2) Per-cluster means
clu_mean = (df.groupBy("cluster")
              .agg(*[F.avg(c).alias(f"{c}_mean") for c in feat]))

# 3) Combine global and cluster stats
joined = clu_mean.crossJoin(glob)

# 4) Compute standardized deltas ~ (cluster_mean - global_mean) / global_sd

exprs = []
for c in feat:
    delta_raw = F.col(f"{c}_mean") - F.col(f"{c}_gmean")
    gsd       = F.col(f"{c}_gsd")
    delta_std = F.when((gsd.isNull()) | (gsd == 0), delta_raw).otherwise(delta_raw / gsd)
    exprs.append(delta_raw.alias(f"{c}_delta"))
    exprs.append(delta_std.alias(f"{c}_delta_std"))

profiles = joined.select("cluster", *[F.col(f"{c}_mean") for c in feat], *exprs)

# 5) Attach human-friendly cluster names and safe names for BI
profiles_named = (profiles
    .join(names, "cluster", "left")
    .withColumn("cluster_name_s", F.regexp_replace("cluster_name", r"[^A-Za-z0-9_]", "_"))
)

# 6) Build a ranked list of “top features” per cluster by |delta_std|



from pyspark.sql import Window


base_long = (profiles_named
    .select(
        "cluster","cluster_name","cluster_name_s",

        F.array(*[F.lit(c) for c in feat]).alias("feature_names"),
        F.array(*[F.col(f"{c}_delta_std") for c in feat]).alias("feature_scores"),

        *[F.col(f"{c}_mean") for c in feat]
    )
    .withColumn("zipped", F.arrays_zip("feature_names","feature_scores"))
    .selectExpr(
        "cluster","cluster_name","cluster_name_s",
        *[f"{c}_mean" for c in feat],
        "explode(zipped) as kv"
    )
    .select(
        "cluster","cluster_name","cluster_name_s",
        *[F.col(f"{c}_mean") for c in feat],
        F.col("kv.feature_names").alias("feature"),
        F.col("kv.feature_scores").alias("score")
    )
    .withColumn("abs_score", F.abs(F.col("score")))
)

# Rank features per cluster by |score|
w = Window.partitionBy("cluster").orderBy(F.desc("abs_score"))
ranked = base_long.withColumn("rn", F.row_number().over(w)).where("rn <= 3")

# Collect top3 back into an array of structs [{feature, score}, ...]
top3 = (ranked
    .groupBy("cluster","cluster_name","cluster_name_s",
             *[F.col(f"{c}_mean") for c in feat])
    .agg(F.collect_list(F.struct(F.col("feature"), F.col("score"))).alias("top3"))
)

# 7) Persist the rich profile table (means + top3 distinctive features)
spark.sql("DROP TABLE IF EXISTS final__minute_cluster_profiles_rich")
top3.write.mode("overwrite").format("delta").saveAsTable("final__minute_cluster_profiles_rich")

print("Saved: final__minute_cluster_profiles_rich")
display(spark.table("final__minute_cluster_profiles_rich"))










StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 22, Finished, Available, Finished)

Saved: final__minute_cluster_profiles_rich


SynapseWidget(Synapse.DataFrame, c1a23b51-1447-4ece-95f3-e032aceefe47)

In [25]:
hour_smooth = spark.table("final__minute_cluster_dist_by_hour_named_smooth")
display(hour_smooth.orderBy("hour_of_day","cluster_name"))



StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 27, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, d0ce7352-90f4-4618-92af-004863552abf)

In [26]:
from pyspark.sql import functions as F

cm = spark.table("final__cm_rowpct")
cm_pivot = (cm.groupBy("label_name")
              .pivot("pred_name")
              .agg(F.first("row_pct"))
              .fillna(0)
              .orderBy("label_name"))
display(cm_pivot)



StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 28, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 31c74606-82ce-431d-ad78-7ed28dced075)

In [27]:
pca = spark.table("final__minute_cluster_pca2_named_sample")
display(pca.select("pc1","pc2","cluster_name_s"))



StatementMeta(, 04c57e1a-5ea2-4ec2-8167-63be6d69d476, 29, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ca8b0e10-6e58-4508-bb0a-2682d4f10472)